# Text Generation using LSTM

This project builds a deep learning model to generate text from Amazon review data using LSTM.

It demonstrates text preprocessing, sequence modeling, and next-word prediction.

In [5]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Embedding
from tensorflow.keras.optimizers import RMSprop



In [3]:
# load dataset
df = pd.read_csv("../data/Amazon_Reviews.csv")

In [4]:
df.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up andâ€¦,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


In [8]:
df.shape

(21214, 9)

In [9]:
df.columns

Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')

In [10]:
# select and clean text data
text_data = df['Review Text']
text_data.head()

0    I registered on the website, tried to order a ...
1    Had multiple orders one turned up and driver h...
2    I informed these reprobates that I WOULD NOT B...
3    I have bought from Amazon before and no proble...
4    If I could give a lower rate I would! I cancel...
Name: Review Text, dtype: object

In [11]:
text_data = text_data.str.lower()
text_data.head()

0    i registered on the website, tried to order a ...
1    had multiple orders one turned up and driver h...
2    i informed these reprobates that i would not b...
3    i have bought from amazon before and no proble...
4    if i could give a lower rate i would! i cancel...
Name: Review Text, dtype: object

In [12]:
text_data = text_data.dropna()
text_data.isnull().sum()

np.int64(0)

In [13]:
# creating tokenizer to convert words into numbers
tokenizer = Tokenizer()

#learn vocabulary from text data
tokenizer.fit_on_texts(text_data)

#convert text into numerical sequences
sequences = tokenizer.texts_to_sequences(text_data)

print(sequences[:5])

[[2, 1991, 17, 1, 279, 202, 3, 39, 5, 1057, 2030, 46, 1, 507, 25, 314, 10, 807, 21, 4, 616, 1, 95, 7, 3324, 8, 52, 3762, 1590, 1260, 1591, 2, 134, 32, 93, 7, 85, 7, 55, 107, 32, 311, 509, 230, 18, 2705, 106, 45, 5, 234, 4, 31, 57, 62, 133, 41, 121, 89, 3763, 2873, 10, 40, 9, 38, 27, 624, 7, 53, 245, 21, 3, 15408, 2528, 152, 59, 787, 307, 5, 329, 1273, 18, 8, 600, 2, 521, 485, 105, 318, 58, 20, 22, 172, 117, 661, 18, 5, 379, 708, 5, 15409, 107, 15410, 13, 544, 1350, 3, 362], [34, 486, 192, 57, 734, 58, 4, 207, 34, 3, 114, 30, 31, 231, 190, 17, 689, 63, 533, 46, 65, 11, 407, 102, 3, 36, 5, 398, 211, 400, 185, 30, 31, 190, 17, 689, 505, 230, 298, 18, 51, 88, 67, 129, 40, 126, 208, 33, 331, 38, 59, 103, 77], [2, 760, 198, 15411, 13, 2, 55, 15, 27, 18, 30, 2, 16, 155, 3, 3057, 5, 1079, 3659, 7, 83, 21, 7, 99, 155, 3, 199, 5, 1748, 2, 83, 32, 2, 97, 15, 212, 9, 30, 2, 16, 5031, 5, 295, 159, 17, 1, 15412, 35, 649, 16, 210, 1600, 81, 62, 319, 2, 2832, 64, 2, 349, 212, 5032, 384, 7, 85, 259, 16

In [14]:
#padding sequences to same length
max_length = 80
padded_sequences = pad_sequences(sequences,maxlen=max_length,padding = 'pre')

# create input-output pairs for next-word prediction
X = []
y = []

for seq in padded_sequences:
    for i in range(1,len(seq)):
        X.append(seq[:i]) # input sequence
        y.append(seq[i])  # next word(target)

# pad input sequences to make them equal length
X = pad_sequences(X,maxlen=max_length,padding = 'pre')

# convert to numpy arrays for model training
X = np.array(X)
y = np.array(y)

X = X[:50000]
y = y[:50000]

print(X.shape, y.shape)

(50000, 80) (50000,)


In [15]:
# build LSTM model

from tensorflow.keras.layers import Dropout
model = Sequential()
vocab_size = len(tokenizer.word_index) + 1
model.add(Embedding(vocab_size, 150))
model.add(LSTM(256))
model.add(Dropout(0.2))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [16]:
# Train model
model.fit(X, y, epochs=5, batch_size=128)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 413s 1s/step - accuracy: 0.2916 - loss: 5.2819
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 385s 983ms/step - accuracy: 0.3103 - loss: 4.6184
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 372s 951ms/step - accuracy: 0.3146 - loss: 4.4857
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 382s 978ms/step - accuracy: 0.3241 - loss: 4.3707
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 374s 955ms/step - accuracy: 0.3316 - loss: 4.2534


In [17]:
# Save trained model
model.save("text_generator_model.h5")

In [21]:
# Top-K sampling function
def sample_top_k(predictions, k=5):
    predictions = predictions.flatten()
    
    top_k_indices = predictions.argsort()[-k:]
    
    top_k_probs = predictions[top_k_indices]
    top_k_probs = top_k_probs / np.sum(top_k_probs)
    
    return np.random.choice(top_k_indices, p=top_k_probs)


# Text generation function
def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=50, padding='pre')

        predicted = model.predict(token_list, verbose=0)

        # use Top-K instead of argmax
        predicted_word_index = sample_top_k(predicted, k=5)

        # convert index → word
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text

In [19]:
# Test model
print(generate_text("this product is", 5))

this product is amazon they have my order


In [20]:
print(generate_text("i love this", 5))
print(generate_text("the quality is", 5))
print(generate_text("this item was", 5))

i love this they have a customer service
the quality is my money of the refund
this item was amazon and a item and


## Conclusion

This project demonstrates text generation using an LSTM-based deep learning model trained on Amazon review data.

The model learns patterns in text and predicts the next word in a sequence to generate sentences.

### Results
- The model is able to generate meaningful text based on input prompts.
- However, due to limitations of LSTM models, the generated text may not always be grammatically perfect or fully coherent.

### Future Improvements
- Use larger datasets for better learning
- Implement Transformer-based models (like GPT) for improved text generation
- Fine-tune hyperparameters for better performance

---

This project provides a foundational understanding of Natural Language Processing and sequence modeling using deep learning.